# DeltaNet v2: Performance Benchmark on Google Colab (Tesla T4)
### Comparing DeltaNet v2 against NVLabs GatedDeltaNet-2 and Flash Linear Attention

> **Environment & Fast Install**: Prebuilt wheels in `wheels/` enable instant installation. If rebuilding from source, the notebook targets **C++20** by default, with automatic fallback to **C++17** and PyTorch 2.4.1.

This notebook benchmarks strictly the Gated DeltaNet recurrence token mixers (without MLP/FFW/SWA overhead):
1. **DeltaNet v2 (Ours)**: Fused CUDA kernels with decoupled k, v gating.
2. **NVLabs GatedDeltaNet-2**: Official NVIDIA research implementation from `NVlabs/GatedDeltaNet-2`.
3. **FLA GatedDeltaNet-2**: Decoupled erase/write linear attention implemented in Flash Linear Attention (`fla.layers.GatedDeltaNet2`).
4. **FLA GatedDeltaNet (v1)**: Standard Gated DeltaNet linear attention layer (`fla.layers.GatedDeltaNet`).
5. **PyTorch Reference**: Unfused PyTorch eager baseline.


## 1. Verify GPU Hardware
Confirm that Google Colab is running with an **NVIDIA Tesla T4** GPU (16 GB VRAM).

In [1]:
!nvidia-smi

Sat Nov 14 17:26:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Repository & Build DeltaNet v2
Clones repository and installs DeltaNet v2.

**Compilation Strategy**:
- **Fast Install**: Installs directly from prebuilt wheel in `wheels/*.whl` if available.
- **Source Build**: If no wheel is present or rebuild is forced, compiles with **C++20** on the active PyTorch environment.
- **Fallback**: If C++20 fails, falls back to PyTorch 2.4.1 and rebuilds with **C++17**.
- **Resource Protection**: Enforces `MAX_JOBS=2` to prevent Colab CPU/RAM exhaustion.

In [2]:
%%bash
# 1. Clone repository with dedicated third_party/cutlass submodule
rm -rf /content/Deltanet
git clone --recursive https://github.com/CBG6682/Varlen-Gated-DeltaNet.git /content/Deltanet
cd /content/Deltanet

# 2. Ensure third_party/cutlass submodule is initialized
git submodule update --init --depth 1 third_party/cutlass || true

# 3. Install build dependencies
pip install -q ninja setuptools wheel einops

# 4. Check for prebuilt wheel for instant installation
INSTALLED=0
if [ "${FORCE_REBUILD:-1}" != "1" ] && ls wheels/*.whl 1> /dev/null 2>&1; then
    echo "===> [Fast Install] Found prebuilt wheel, installing directly..."
    if pip install wheels/*.whl; then
        echo "===> [SUCCESS] DeltaNet v2 installed instantly from prebuilt wheel!"
        INSTALLED=1
    fi
fi

# 5. If prebuilt wheel was not installed or rebuild is forced, compile with C++20 (fallback C++17)
if [ "$INSTALLED" -eq 0 ]; then
    export MAX_JOBS=2
    export CXX_STANDARD=c++20
    echo "===> [1/2] Attempting compilation with C++20..."
    if pip install -v . --no-build-isolation; then
        echo "===> [SUCCESS] Built DeltaNet v2 with C++20!"
    else
        echo "===> [FALLBACK] C++20 compilation failed. Falling back to C++17 with PyTorch 2.4.1..."
        pip install -q torch==2.4.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
        export CXX_STANDARD=c++17
        export FORCE_REBUILD=1
        pip install -v . --no-build-isolation
        echo "===> [SUCCESS] Built DeltaNet v2 with C++17!"
    fi
fi

# Copy compiled .so into local package directory for in-tree imports
cp build/lib.*/deltanet_v2/_C*.so deltanet_v2/ 2>/dev/null || true


Submodule path 'third_party/cutlass': checked out 'b2f249acd20392beeb16f5e5a31f4f1a4e38b3bd'
===> [Fast Install] Found prebuilt wheel, installing directly...
Processing ./wheels/deltanet_v2-2.0.0-py3-none-any.whl
===> [SUCCESS] DeltaNet v2 installed instantly from prebuilt wheel!


Cloning into '/content/Deltanet'...
Submodule 'third_party/cutlass' (https://github.com/NVIDIA/cutlass.git) registered for path 'third_party/cutlass'
Cloning into '/content/Deltanet/third_party/cutlass'...


## 3. (Optional) Package Standalone Wheel (.whl)
Packages the compiled extension into a clean `.whl` distribution strictly restricted to `deltanet_v2/` (excluding foreign container files) and downloads it.

In [3]:
# ==============================================================================
# Package clean Wheel (.whl) containing only deltanet_v2
# ==============================================================================
import os
import glob
import zipfile
from google.colab import files

%cd /content/Deltanet

# Clean previous build directories
!rm -rf dist build *.egg-info

# Build wheel package (setup.py strictly restricts packages to deltanet_v2)
!python setup.py bdist_wheel

# Inspect wheel contents to verify no outside container files were included
wheel_files = glob.glob("dist/*.whl")
if wheel_files:
    target_whl = wheel_files[0]
    print("\n" + "=" * 60)
    print(f"VERIFYING WHEEL CONTENTS: {os.path.basename(target_whl)}")
    print("=" * 60)
    with zipfile.ZipFile(target_whl, "r") as z:
        for member in z.namelist():
            print(f"  - {member}")
    print("=" * 60)

    # Trigger download to local machine
    print(f"\n>> Downloading wheel: {os.path.basename(target_whl)}...")
    files.download(target_whl)
else:
    print("[!] No wheel file found in /content/Deltanet/dist/!")


/content/Deltanet
[setup.py] Setting default MAX_JOBS=2 to prevent CPU/RAM exhaustion
[setup.py] Pre-compiled extension found: _C.cpython-313-x86_64-linux-gnu.so. Skipping compilation.
[setup.py] (Set FORCE_REBUILD=1 to recompile from source).
running bdist_wheel
running build
running build_py
creating build/lib/deltanet_v2
copying deltanet_v2/model.py -> build/lib/deltanet_v2
copying deltanet_v2/ops.py -> build/lib/deltanet_v2
copying deltanet_v2/__init__.py -> build/lib/deltanet_v2
copying deltanet_v2/_C.cpython-313-x86_64-linux-gnu.so -> build/lib/deltanet_v2
/usr/local/lib/python3.13/dist-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This deprecation is overdue, please update your projec

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Install Baseline Libraries
Installs **Flash Linear Attention** (`fla`) and clones **NVIDIA NVLabs GatedDeltaNet-2** (`NVlabs/GatedDeltaNet-2`).
Neutralizes `lit_gpt/__init__.py` to allow direct standalone import of `GatedDeltaNet2` without `lightning` or `flash-attn` errors.

In [4]:
%%bash
# 1. Install Flash Linear Attention with pinned commit hash
pip install -q git+https://github.com/sustcsonglin/flash-linear-attention.git@2705ab8ee9a9086bcfa6d079858e4281e01215a1 --no-build-isolation

# 2. Clone official NVIDIA NVLabs GatedDeltaNet-2 repository with pinned commit hash
rm -rf /content/gateddeltanet-2
git clone https://github.com/NVlabs/GatedDeltaNet-2.git /content/gateddeltanet-2
cd /content/gateddeltanet-2 && git checkout de8476de0fecae08fc16f0bf3154900244c61c73 && cd /content/Deltanet

# 3. Neutralize lit_gpt/__init__.py to allow direct import of GatedDeltaNet2 without lightning / flash-attn errors
echo "# Neutralized for standalone GatedDeltaNet2" > /content/gateddeltanet-2/lit_gpt/__init__.py


Cloning into '/content/gateddeltanet-2'...


## 5. Numerical Correctness Sanity Test
Verifies numerical consistency between:
- **Inference decode step ($L = 1$)**: CUDA Kernel (`use_kernel=True`) vs PyTorch Reference (`use_kernel=False`).
- **Varlen training forward pass**: CUDA Kernel varlen (`use_kernel=True`) vs PyTorch varlen reference (`use_kernel=False, varlen=True`).

In [5]:
!python /content/Deltanet/tests/test_correctness.py

HARDWARE & RUNTIME ENVIRONMENT
PyTorch Version  : 2.14.1+cu130
CUDA Available   : True
Device Name      : Tesla T4
Compute Arch     : SM7.5
Total VRAM       : 14.56 GB
Multi-Processors : 40

########################################################################
# RUNNING NUMERICAL CORRECTNESS TESTS
########################################################################

--> Checking Autoregressive Decode Step (L = 1) Numerical Consistency...
    Output Tensor   : max_diff = 0.000488, cos_sim = 1.000000
    Recurrent State : max_diff = 0.001953, cos_sim = 1.000000
    [PASS] Inference kernel matches PyTorch reference perfectly!

--> Checking Varlen Training Forward Pass Numerical Consistency...
    Comparing: Kernel Varlen (use_kernel=True) vs Reference (use_kernel=False, varlen=True)...
    Sequence Lengths: [128, 256, 128] (Total tokens: 512, Batch: 3)
    Output Tensor   : max_diff = 0.089417, cos_sim = 0.998201
    Recurrent State : max_diff = 0.056152, cos_sim = 0.996383
    [PA

## 6. Benchmark: Autoregressive Inference (L = 1)
Measures single-step decode latency (ms/step), generation throughput (tokens/sec), operation count (GFLOPs), compute throughput (TFLOP/s), memory throughput (GB/s), and peak VRAM across batch sizes $B \in [1, 16, 64, 256, 1024, 2048]$ using an interactive notebook progress bar.
Evaluates strictly the Gated DeltaNet token-mixer blocks without MLP/FFW overhead.

In [6]:
!python -u /content/Deltanet/tests/bench_inference.py --dim 768 --q-heads 8 --kv-heads 8 --batches 1,16,64,256,1024,2048 --warmup 5 --iters 20 --save-json /content/Deltanet/assets/infer_results.json


HARDWARE & RUNTIME ENVIRONMENT
PyTorch Version  : 2.14.1+cu130
CUDA Available   : True
Device Name      : Tesla T4
Compute Arch     : SM7.5
Total VRAM       : 14.56 GB
Multi-Processors : 40

BASELINE AVAILABILITY
[-] DeltaNet v2 (Ours)                 : INSTALLED
[-] DeltaNet v2 (PyTorch Reference)    : INSTALLED
[-] NVLabs GatedDeltaNet-2 (Official)  : INSTALLED
[-] FLA GatedDeltaNet-2 (FLA Library)  : INSTALLED
[-] FLA GatedDeltaNet (v1)             : INSTALLED


########################################################################
# BENCHMARK: SINGLE-STEP AUTOREGRESSIVE INFERENCE (L = 1)
########################################################################

Config: d_model=768, num_q_heads=8, num_kv_heads=8, d_k=64, d_v=64, use_short_conv=False, dtype=torch.float16, device=cuda:0
Batch sizes to evaluate: [1, 16, 64, 256, 1024, 2048]

Initializing DeltaNet v2...
Initializing NVLabs GatedDeltaNet-2 (Official NVIDIA reference)...
Initializing FLA GatedDeltaNet-2 (Flash Linear Att

## 7. Benchmark: Training Pass (Forward + Backward)
Measures step latency (ms/step), theoretical operation count (GFLOPs), compute throughput (TFLOP/s), and peak VRAM across sequence lengths $L \in [512, 1024, 2048, 4096]$ using an interactive notebook progress bar.
Evaluates strictly the Gated DeltaNet token-mixer blocks without MLP/FFW overhead.

In [7]:
!python -u /content/Deltanet/tests/bench_training.py --dim 768 --q-heads 8 --kv-heads 8 --batch-size 4 --seq-lens 512,1024,2048,4096 --warmup 3 --iters 10 --save-json /content/Deltanet/assets/train_results.json


HARDWARE & RUNTIME ENVIRONMENT
PyTorch Version  : 2.14.1+cu130
CUDA Available   : True
Device Name      : Tesla T4
Compute Arch     : SM7.5
Total VRAM       : 14.56 GB
Multi-Processors : 40

BASELINE AVAILABILITY
[-] DeltaNet v2 (Ours)                 : INSTALLED
[-] DeltaNet v2 (PyTorch Reference)    : INSTALLED
[-] NVLabs GatedDeltaNet-2 (Official)  : INSTALLED
[-] FLA GatedDeltaNet-2 (FLA Library)  : INSTALLED
[-] FLA GatedDeltaNet (v1)             : INSTALLED


########################################################################
# BENCHMARK: TRAINING PASS (FORWARD + BACKWARD)
########################################################################

Config: d_model=768, num_q_heads=8, num_kv_heads=8, d_k=64, d_v=64, chunk_size=64, batch_size=4, use_short_conv=False, dtype=torch.float16, device=cuda:0
Sequence lengths to evaluate: [512, 1024, 2048, 4096]

Initializing DeltaNet v2...
Initializing NVLabs GatedDeltaNet-2 (Official NVIDIA chunkwise parallel)...
Initializing FLA Gated

## 8. Plot Open-Source Baselines Chart
Generates `assets/benchmark_opensource_baselines.png` from the inference and training benchmark JSON outputs.


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

with open("/content/Deltanet/assets/infer_results.json") as f:
    inf_rows = json.load(f)
with open("/content/Deltanet/assets/train_results.json") as f:
    trn_rows = json.load(f)

methods = [
    "PyTorch Reference (Unfused)",
    "FLA GatedDeltaNet (v1)",
    "FLA GatedDeltaNet-2",
    "NVLabs GatedDeltaNet-2",
    "DeltaNet v2 (Ours)",
]
colors = ["#7f8c8d", "#3498db", "#9b59b6", "#e67e22", "#e74c3c"]

batches = sorted({r["batch_size"] for r in inf_rows})
seq_lens = sorted({r["seq_len"] for r in trn_rows})

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("DeltaNet v2 vs Open-Source Baselines (Tesla T4 SM75, FP16)", fontsize=14, fontweight="bold")

ax = axes[0, 0]
for m, c in zip(methods, colors):
    vals = [next((r["throughput_tok_s"] for r in inf_rows if r["batch_size"] == b and r["method"] == m), None) for b in batches]
    if any(v is not None for v in vals):
        ax.plot([str(b) for b in batches], [v or 0 for v in vals], marker="o", linewidth=2.2 if "Ours" in m else 1.5, label=m, color=c)
ax.set_title("Autoregressive Decode (L=1): Throughput (tok/s)")
ax.set_xlabel("Batch Size (B)")
ax.set_ylabel("Tokens / sec")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

ax = axes[0, 1]
for m, c in zip(methods, colors):
    vals = [next((r["latency_ms"] for r in inf_rows if r["batch_size"] == b and r["method"] == m), None) for b in batches]
    if any(v is not None for v in vals):
        ax.plot([str(b) for b in batches], [v or 0 for v in vals], marker="o", linewidth=2.2 if "Ours" in m else 1.5, label=m, color=c)
ax.set_title("Autoregressive Decode (L=1): Latency (ms)")
ax.set_xlabel("Batch Size (B)")
ax.set_ylabel("Latency (ms)")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

ax = axes[1, 0]
x = np.arange(len(seq_lens))
w = 0.15
for idx, (m, c) in enumerate(zip(methods, colors)):
    vals = [next((r["fwd_bwd_ms"] for r in trn_rows if r["seq_len"] == L and r["method"] == m), 0) or 0 for L in seq_lens]
    if any(vals):
        ax.bar(x + (idx - 2) * w, vals, width=w, label=m, color=c)
ax.set_title("Training (Fwd+Bwd, B=4): Step Latency (ms)")
ax.set_xticks(x)
ax.set_xticklabels([str(L) for L in seq_lens])
ax.set_xlabel("Sequence Length (L)")
ax.set_ylabel("Latency (ms)")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(fontsize=8)

ax = axes[1, 1]
for idx, (m, c) in enumerate(zip(methods, colors)):
    vals = [next((r["peak_mb"] for r in trn_rows if r["seq_len"] == L and r["method"] == m), 0) or 0 for L in seq_lens]
    if any(vals):
        ax.bar(x + (idx - 2) * w, vals, width=w, label=m, color=c)
ax.set_title("Training (Fwd+Bwd, B=4): Peak VRAM (MB)")
ax.set_xticks(x)
ax.set_xticklabels([str(L) for L in seq_lens])
ax.set_xlabel("Sequence Length (L)")
ax.set_ylabel("Peak Allocated VRAM (MB)")
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
out_path = "/content/Deltanet/assets/benchmark_opensource_baselines.png"
plt.savefig(out_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")


## 9. Benchmark & Plot End-to-End `DeltaBlock` (`torch.compile`)
Evaluates full `DeltaBlock` (projections, RMSNorm, gating, MLP) across `torch-eager`, `torch-compiled`, and `kernel-compiled`, and saves `assets/benchmark_compiled_e2e.png`.


In [ ]:
import gc
import matplotlib.pyplot as plt
import numpy as np
import torch
from deltanet_v2 import DeltaBlock

device = torch.device("cuda")
D_MODEL, HQ, HKV, HIDDEN = 768, 8, 4, 768 * 4

def bench_ms(fn, warmup=5, iters=15):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(iters):
        fn()
    e.record()
    torch.cuda.synchronize()
    return s.elapsed_time(e) / iters

def make_block():
    return DeltaBlock(D_MODEL, HQ, HKV, 64, 64, 64, hidden_dim=HIDDEN).to(device).half()

decode_batches = [16, 64, 256, 1024]
dec_res = {"torch-eager": [], "torch-compiled": [], "kernel-compiled": []}

for B in decode_batches:
    m_e = make_block().eval()
    m_tc = torch.compile(make_block().eval(), backend="inductor")
    m_kc = torch.compile(make_block().eval(), backend="inductor")
    x = torch.randn(B, 1, D_MODEL, device=device, dtype=torch.float16)
    p = torch.zeros(B, HKV, 64, 64, device=device, dtype=torch.float16)

    with torch.no_grad():
        t_e = bench_ms(lambda: m_e(x, past_states=p.clone(), use_kernel=False))
        t_tc = bench_ms(lambda: m_tc(x, past_states=p.clone(), use_kernel=False))
        t_kc = bench_ms(lambda: m_kc(x, past_states=p.clone(), use_kernel=True))
    dec_res["torch-eager"].append(B * 1000.0 / t_e)
    dec_res["torch-compiled"].append(B * 1000.0 / t_tc)
    dec_res["kernel-compiled"].append(B * 1000.0 / t_kc)
    print(f"[Decode B={B}] eager={t_e:.2f}ms ({dec_res['torch-eager'][-1]:,.0f} tok/s) | "
          f"torch-comp={t_tc:.2f}ms ({dec_res['torch-compiled'][-1]:,.0f} tok/s) | "
          f"kernel-comp={t_kc:.2f}ms ({dec_res['kernel-compiled'][-1]:,.0f} tok/s)")
    del m_e, m_tc, m_kc
    gc.collect(); torch.cuda.empty_cache()

train_batches = [4, 8, 16]
trn_res = {"torch-eager": [], "torch-compiled": [], "kernel-compiled": []}
trn_mem = {"torch-eager": [], "torch-compiled": [], "kernel-compiled": []}

for B in train_batches:
    L = 1024
    cu = torch.arange(0, (B + 1) * L, L, dtype=torch.int32, device=device)
    for mode, use_k, do_comp in [("torch-eager", False, False), ("torch-compiled", False, True), ("kernel-compiled", True, True)]:
        try:
            gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            blk = make_block().train()
            fn = torch.compile(blk, backend="inductor") if do_comp else blk
            x = torch.randn(B * L, D_MODEL, device=device, dtype=torch.float16, requires_grad=True) if use_k else torch.randn(B, L, D_MODEL, device=device, dtype=torch.float16, requires_grad=True)
            def step():
                blk.zero_grad(set_to_none=True)
                x.grad = None
                out, _ = fn(x, cu_seqlens=cu, use_kernel=True) if use_k else fn(x, use_kernel=False)
                out.sum().backward()
            lat = bench_ms(step, warmup=3, iters=8)
            peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
            trn_res[mode].append((B * L * 1000.0) / lat)
            trn_mem[mode].append(peak_gb)
            print(f"[Train B={B}] {mode}: {lat:.1f}ms | {trn_res[mode][-1]:,.0f} tok/s | {peak_gb:.2f} GB")
        except torch.cuda.OutOfMemoryError:
            trn_res[mode].append(0.0)
            trn_mem[mode].append(0.0)
            print(f"[Train B={B}] {mode}: OOM")
        gc.collect(); torch.cuda.empty_cache()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
fig.suptitle("End-to-End DeltaBlock with torch.compile (Tesla T4 SM75, FP16)", fontsize=13, fontweight="bold")
colors = {"torch-eager": "#7f8c8d", "torch-compiled": "#3498db", "kernel-compiled": "#e74c3c"}

for m in ["torch-eager", "torch-compiled", "kernel-compiled"]:
    axes[0].plot([str(b) for b in decode_batches], dec_res[m], marker="o", linewidth=2, label=m, color=colors[m])
axes[0].set_title("Decode Throughput (L=1)")
axes[0].set_xlabel("Batch Size")
axes[0].set_ylabel("Tokens / sec")
axes[0].grid(True, alpha=0.3); axes[0].legend(fontsize=8)

xb = np.arange(len(train_batches))
w = 0.25
for idx, m in enumerate(["torch-eager", "torch-compiled", "kernel-compiled"]):
    axes[1].bar(xb + (idx - 1) * w, trn_res[m], width=w, label=m, color=colors[m])
    axes[2].bar(xb + (idx - 1) * w, trn_mem[m], width=w, label=m, color=colors[m])
axes[1].set_title("Training Throughput (L=1024)")
axes[1].set_xticks(xb); axes[1].set_xticklabels([f"B={b}" for b in train_batches]); axes[1].set_ylabel("Tokens / sec"); axes[1].grid(True, axis="y", alpha=0.3)
axes[2].set_title("Training Peak VRAM (L=1024)")
axes[2].set_xticks(xb); axes[2].set_xticklabels([f"B={b}" for b in train_batches]); axes[2].set_ylabel("Peak VRAM (GB)"); axes[2].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
out_path = "/content/Deltanet/assets/benchmark_compiled_e2e.png"
plt.savefig(out_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")
